In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from calculate_param_models import calculate_param_models
from calculate_metrics import calculate_metrics
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier


ModuleNotFoundError: No module named 'plot_classification_surface'

In [ ]:
penguins = pd.read_csv('penguins.csv')
penguins

In [ ]:

penguins = penguins.dropna()
X = penguins.drop(columns='Species')
features = X.columns.to_list()
y = penguins['Species']
pd.Series(y).value_counts(normalize = True)
penguins.info()

In [ ]:
X.describe()

In [ ]:
penguins.isnull().sum()

In [ ]:
import seaborn as sns
for col in features:
    plt.figure(figsize=(12, 2.5))
    sns.histplot(penguins.loc[penguins['Species']==0, col], kde=True, color='green', label='Species 0')
    sns.histplot(penguins.loc[penguins['Species']==1, col], kde=True, color='red', label='Species 1')
    sns.histplot(penguins.loc[penguins['Species']==2, col], kde=True, color='blue', label='Species 2')
    plt.legend(loc='upper right')
    plt.show()

Tutaj bym się już zatrzymał zbiory są dość dobrze rozdzielone więc jakieś tree powinno sobie poradzić z tym zbiorem. Pewnym problemem mogą okazać się połączenia zbiorów na Culmen oraz nie duża liczba recordów w zbiorze.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print ('Treningowe obserwacje: %d\nTestowe obserwacje: %d' % (X_train.shape[0], X_test.shape[0]))
calculate_param_models(X_train,y_train)

In [ ]:
print('KNN')
knn_model = KNeighborsClassifier(n_neighbors=3,p=1)
knn_model.fit(X_train,y_train)
calculate_metrics(knn_model,'KNN',X_test,y_test)
print('DecisionTreeClassifier')
dtr_model = DecisionTreeClassifier(max_depth=5,min_samples_leaf=2)
dtr_model.fit(X_train,y_train)
calculate_metrics(dtr_model,'DecisionTreeClassifier',X_test,y_test)
print('RandomForestClassifier')
rfr_model = RandomForestClassifier(max_depth=5,min_samples_leaf=3, n_estimators=1000)
rfr_model.fit(X_train,y_train)
calculate_metrics(rfr_model,'RandomForestClassifier',X_test,y_test)
print('SVC')
svc_model = SVC(C=1, kernel='linear',probability=True)
svc_model.fit(X_train,y_train)
calculate_metrics(svc_model,'SVC',X_test,y_test)
print('AdaBoostClassifier')
# model_adaboost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50)
ada_model = AdaBoostClassifier(learning_rate=0.1, n_estimators=200)
ada_model.fit(X_train,y_train)
calculate_metrics(ada_model,'AdaBoostClassifier',X_test,y_test)

model_voting = VotingClassifier(estimators=[('Tree', dtr_model),
                                            ('Random Forest', rfr_model),
                                            ('AdaBoost', ada_model),
                                            ('SVC',svc_model),
                                            ('KNN',knn_model)
                                            ],
                                voting='soft')

print('Model Voting')
model_voting.fit(X_train, y_train)
calculate_metrics(model_voting,'Model Voting',X_test,y_test)

